# Alerta de Risco da Cadeia — Previsibilidade & Cascateamento

**Data de referência:** parametrizável (default = hoje)  
**Fonte:** `analytics/plan_freeze_rate/sql/kr1_evolucao.sql` + `plano_vs_atual.sql`

Sistema operacional de acompanhamento de risco baseado nas premissas do diagnóstico de cascateamento (`analyses/cascateamento_fornecedores/README.md`).

Quatro crivos de alerta:

| Seção | Pergunta | Janela | Critério de destaque |
|---|---|---|---|
| 1. Cascateamento histórico | Quais fornecedores propagam problemas entre ciclos? | Histórico completo | Cascade > 50% |
| 2. Sinal de risco prospectivo | Quais fornecedores tiveram alta alteração nos meses recentes? | M-2 e M-1 | pct M-1 > 60% **ou** pct M-2 > 70% |
| 3. Janela operacional | Quais ciclos ativos estão alterando dentro do lead time? | M e M+1 | pct_ext_any > 20% **e** antecipação < 45 dias |
| 4. Aumento em M | Houve qualquer evento de aumento week-over-week no ciclo corrente? | Histórico do ciclo M | qualquer delta pct_ext_any > 0 |

> **Convenção de meses:** M = mês de referência (default = mês corrente). `mes_alvo` é o mês dominante de `baseline_dt_planned` por ciclo (mesmo critério do diagnóstico).

In [158]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import datetime
from pathlib import Path

In [159]:
from google.cloud import bigquery

client = bigquery.Client(project="insider-data-lake")

# ----- Parâmetros -----
REF_DATE = pd.to_datetime("today").normalize()           # data de referência
REF_MONTH = REF_DATE.to_period('M')                       # M = mês de referência
LEAD_TIME_DIAS = 45                                       # lead time produtivo de referência

# Critérios de destaque (todos baseados em pct_ext_any)
CASCADE_THRESHOLD_HIST = 50      # %, Seção 1
PCT_M_1_THRESHOLD = 60           # %, Seção 2
PCT_M_2_THRESHOLD = 70           # %, Seção 2
PCT_ATIVO_THRESHOLD = 20         # %, Seção 3

# Threshold para definir "ciclo problemático" no cálculo de cascade (mesmo do diagnóstico)
PEAK_THRESHOLD_CASCADE = 30      # %

today = REF_DATE.strftime("%Y%m%d")
sql_path = '../../analytics/plan_freeze_rate/sql/'
output_path = '../../outputs/'
CICLOS_EXCLUIDOS = ['C012026']
WATERFALL_CYCLE_TYPE = 'Base'

print(f"Data de referência : {REF_DATE.date()}")
print(f"M  (mês corrente)  : {REF_MONTH}")
print(f"M-1                : {REF_MONTH - 1}")
print(f"M-2                : {REF_MONTH - 2}")
print(f"M+1                : {REF_MONTH + 1}")

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Data de referência : 2026-04-30
M  (mês corrente)  : 2026-04
M-1                : 2026-03
M-2                : 2026-02
M+1                : 2026-05


# 1. Carregamento dos dados

In [160]:
def query_to_dataframe(query):
    return client.query(query).result().to_dataframe()

def read_sql_file(file_path):
    with open(file_path, 'r') as f:
        return f.read()

def convert_sql_to_df(file_path):
    return query_to_dataframe(read_sql_file(file_path))


def derive_mes_alvo(df_plano):
    """Mes_alvo = mês dominante de baseline_dt_planned por ciclo (mesmo critério do KR1)."""
    mes_alvo = (
        df_plano.groupby('cycle_name')
        .apply(lambda g: g.groupby(pd.to_datetime(g['baseline_dt_planned']).dt.to_period('M'))['baseline_planned_qty']
               .sum().idxmax())
        .reset_index()
    )
    mes_alvo.columns = ['cycle_name', 'mes_alvo']
    return mes_alvo


def load_data(sql_path, ciclos_excluidos):
    print("Carregando plano_vs_atual.sql ...")
    df_plano = convert_sql_to_df(sql_path + 'plano_vs_atual.sql')
    df_plano = df_plano[~df_plano['cycle_name'].isin(ciclos_excluidos)].copy()

    print("Carregando kr1_evolucao.sql ...")
    df_evolucao = convert_sql_to_df(sql_path + 'kr1_evolucao.sql')
    df_evolucao = df_evolucao[~df_evolucao['cycle_name'].isin(ciclos_excluidos)].copy()

    mes_alvo = derive_mes_alvo(df_plano)
    df_evolucao = df_evolucao.merge(mes_alvo, on='cycle_name', how='left')
    df_plano = df_plano.merge(mes_alvo, on='cycle_name', how='left')

    df_evolucao['snapshot_week'] = pd.to_datetime(df_evolucao['snapshot_week'])
    df_evolucao['mes_alvo_str'] = df_evolucao['mes_alvo'].astype(str)

    print(f"  plano:    {len(df_plano):,} linhas | {df_plano['cycle_name'].nunique()} ciclos")
    print(f"  evolução: {len(df_evolucao):,} linhas | {df_evolucao['snapshot_week'].nunique()} semanas")
    return df_plano, df_evolucao


df_plano, df_evolucao = load_data(sql_path, CICLOS_EXCLUIDOS)

Carregando plano_vs_atual.sql ...


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Carregando kr1_evolucao.sql ...


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_47931/317389198.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.groupby(pd.to_datetime(g['baseline_dt_planned']).dt.to_period('M'))['baseline_planned_qty']


  plano:    53,696 linhas | 212 ciclos
  evolução: 60,264 linhas | 25 semanas


In [161]:
# Filtra ciclos Base (mesma convenção do diagnóstico)
df_evolucao_base = df_evolucao[df_evolucao['cycle_type'] == 'Base'].copy()
print(f"Ciclos Base na evolução: {df_evolucao_base['cycle_name'].nunique()}")
df_evolucao_base.head()

Ciclos Base na evolução: 13


,snapshot_week,cycle_name,cycle_type,supplier_name,product_name,vol_original,vol_int_date,vol_int_cancel,vol_int_grade,vol_int_any,vol_ext_cancel,vol_ext_date_rev,vol_ext_any,mes_alvo,mes_alvo_str
1505,2025-11-10,C012026B,Base,ART LIVRE,Daily T-shirt Masculino,11868,0,0,0,0,0,0,0,2026-01,2026-01
1506,2025-11-10,C012026B,Base,ART LIVRE,Tech T-shirt Gola V Feminino,4594,0,0,0,0,0,0,0,2026-01,2026-01
1507,2025-11-10,C012026B,Base,ART LIVRE,Tech T-shirt Heavy Masculino,4288,0,0,0,0,4288,0,4288,2026-01,2026-01
1508,2025-11-10,C012026B,Base,ART LIVRE,Tech T-shirt Heavy Slim Masculino,4065,0,0,0,0,4065,0,4065,2026-01,2026-01
1509,2025-11-10,C012026B,Base,ART LIVRE,Tech T-shirt Long Sleeve Feminino,5382,0,0,0,0,1066,0,1066,2026-01,2026-01


# 2. Construção das métricas

Para cada `(supplier, mes_alvo)` calculamos:
- **peak `pct_ext_any`** ao longo da vida do ciclo (high-water mark) — base para cascade e para o snapshot histórico de M-1/M-2
- **`pct_ext_any` corrente** (último snapshot disponível) — para os alertas operacionais de M/M+1
- **antecipação** = dias entre `snapshot_week` em que `pct_ext_any ≥ 10%` e o início de `mes_alvo`

In [162]:
def build_supplier_mes_alvo_metrics(df_evol):
    """Por (supplier, mes_alvo): peak, current pct_ext_any, antecipação."""
    g = (df_evol.groupby(['supplier_name', 'mes_alvo', 'snapshot_week'])
         .agg(vol_original=('vol_original', 'sum'),
              vol_ext_any=('vol_ext_any', 'sum'))
         .reset_index())
    g['pct_ext_any'] = np.where(g['vol_original'] > 0,
                                g['vol_ext_any'] / g['vol_original'] * 100, 0)

    out = []
    for (sup, ma), grp in g.groupby(['supplier_name', 'mes_alvo']):
        grp = grp.sort_values('snapshot_week')
        vol_orig = grp['vol_original'].max()
        peak = grp['pct_ext_any'].max()
        current = grp.iloc[-1]['pct_ext_any']
        last_snap = grp.iloc[-1]['snapshot_week']

        # Antecipação: 1ª semana em que pct_ext_any ≥ 10%
        cross = grp[grp['pct_ext_any'] >= 10]
        if len(cross) > 0:
            first_signal = cross.iloc[0]['snapshot_week']
            mes_alvo_start = pd.Period(ma, freq='M').start_time
            antecip = (mes_alvo_start - first_signal).days
        else:
            antecip = np.nan

        out.append({
            'supplier_name': sup, 'mes_alvo': ma,
            'vol_original': vol_orig,
            'peak_pct_ext_any': round(peak, 1),
            'current_pct_ext_any': round(current, 1),
            'last_snapshot': last_snap,
            'antecipacao_d': antecip,
        })
    return pd.DataFrame(out)


supplier_mes = build_supplier_mes_alvo_metrics(df_evolucao_base)
print(f"Pares (supplier × mes_alvo): {len(supplier_mes):,}")
supplier_mes.head()

Pares (supplier × mes_alvo): 221


,supplier_name,mes_alvo,vol_original,peak_pct_ext_any,current_pct_ext_any,last_snapshot,antecipacao_d
0,ABBA,2026-03,9500,54.2,13.2,2026-04-27,13.0
1,ABBA,2026-04,11747,45.4,24.2,2026-04-27,65.0
2,ABBA,2026-05,8951,19.7,19.7,2026-04-27,18.0
3,ABBA,2026-06,11646,12.9,12.9,2026-04-27,49.0
4,ABBA,2026-07,11715,0.0,0.0,2026-04-27,NaN


In [163]:
# ----- Funções auxiliares de tabelas e visualizações -----
def build_cascade_table(supplier_mes, threshold=30):
    """Cascade % por supplier, contando pares consecutivos onde M já era problemático."""
    rows = []
    for sup, grp in supplier_mes.groupby('supplier_name'):
        grp = grp.sort_values('mes_alvo').reset_index(drop=True)
        # Pares M, M+1 consecutivos
        n_pairs_problem_M = 0
        n_pairs_cascade = 0
        vol_total = grp['vol_original'].sum()
        peak_pct_medio = ((grp['peak_pct_ext_any'] * grp['vol_original']).sum()
                          / vol_total) if vol_total > 0 else np.nan

        for i in range(len(grp) - 1):
            m = grp.iloc[i]['mes_alvo']
            m1 = grp.iloc[i+1]['mes_alvo']
            # Apenas pares de meses consecutivos
            if (m1 - m).n != 1:
                continue
            peak_m = grp.iloc[i]['peak_pct_ext_any']
            peak_m1 = grp.iloc[i+1]['peak_pct_ext_any']
            if peak_m > threshold:
                n_pairs_problem_M += 1
                if peak_m1 > threshold:
                    n_pairs_cascade += 1

        cascade_pct = (n_pairs_cascade / n_pairs_problem_M * 100) if n_pairs_problem_M > 0 else np.nan
        rows.append({
            'supplier_name': sup,
            'vol_total': int(vol_total),
            'n_mes_alvo': len(grp),
            'pct_ext_medio': round(peak_pct_medio, 1) if pd.notna(peak_pct_medio) else np.nan,
            'n_pares_M_problematicos': n_pairs_problem_M,
            'cascade_pct': round(cascade_pct, 1) if pd.notna(cascade_pct) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(['cascade_pct', 'vol_total'],
                                          ascending=[False, False],
                                          na_position='last')


def style_cascade_hist(df, threshold=50):
    """Pandas Styler: heatmap em cascade_pct + destaque negativo > threshold."""
    df_show = df.copy()
    df_show['vol_total'] = df_show['vol_total'].apply(lambda x: f"{x:,}".replace(",", "."))
    df_show.columns = ['Fornecedor', 'Vol. total (pcs)', 'N° mes_alvo',
                       '% EXT médio (peak ponderado)',
                       'N° pares M problemáticos', 'Cascade %']

    def highlight_cascade(row):
        styles = [''] * len(row)
        c = row['Cascade %']
        if pd.notna(c) and c > threshold:
            styles = ['background-color: #ffe5e5; font-weight: bold;'] * len(row)
        return styles

    styled = (df_show.style
              .apply(highlight_cascade, axis=1)
              .background_gradient(subset=['Cascade %'], cmap='Reds', vmin=0, vmax=100)
              .background_gradient(subset=['% EXT médio (peak ponderado)'],
                                   cmap='YlOrRd', vmin=0, vmax=100)
              .format({'Cascade %': '{:.0f}%',
                       '% EXT médio (peak ponderado)': '{:.0f}%'},
                      na_rep='—')
              .set_caption(f"Cascateamento histórico — destaque vermelho: cascade > {threshold}%")
              .set_table_styles([{'selector': 'caption',
                                   'props': [('caption-side', 'top'),
                                             ('font-size', '14px'),
                                             ('font-weight', 'bold'),
                                             ('padding', '8px')]}]))
    return styled


def build_pct_recente_table(supplier_mes, ref_month):
    """Pivot: supplier × {M-2_pct, M-1_pct} usando peak_pct_ext_any."""
    m1 = ref_month - 1
    m2 = ref_month - 2

    rec = supplier_mes[supplier_mes['mes_alvo'].isin([m2, m1])].copy()
    rec['mes_label'] = rec['mes_alvo'].apply(lambda p: f'M-1 ({p})' if p == m1 else f'M-2 ({p})')

    pivot_pct = rec.pivot_table(index='supplier_name', columns='mes_label',
                                values='peak_pct_ext_any', aggfunc='first')
    pivot_vol = rec.pivot_table(index='supplier_name', columns='mes_label',
                                values='vol_original', aggfunc='first')
    # Vol total dos dois meses para ranqueamento
    vol_total = pivot_vol.sum(axis=1).rename('vol_total_M-2_M-1')

    out = pivot_pct.join(vol_total).sort_values('vol_total_M-2_M-1', ascending=False)
    return out, m1, m2


def style_pct_recente(df, m1, m2, thr_m1=60, thr_m2=70):
    """Heatmap M-1/M-2 com destaque negativo nos critérios."""
    col_m1 = f'M-1 ({m1})'
    col_m2 = f'M-2 ({m2})'
    df_show = df.copy()
    # Garante existência das colunas
    for c in [col_m2, col_m1]:
        if c not in df_show.columns:
            df_show[c] = np.nan
    df_show = df_show[[col_m2, col_m1, 'vol_total_M-2_M-1']]
    df_show.columns = [f'% EXT M-2 ({m2})', f'% EXT M-1 ({m1})', 'Vol. M-2+M-1 (pcs)']

    def highlight_row(row):
        v_m1 = row[f'% EXT M-1 ({m1})']
        v_m2 = row[f'% EXT M-2 ({m2})']
        flag = (pd.notna(v_m1) and v_m1 > thr_m1) or (pd.notna(v_m2) and v_m2 > thr_m2)
        return ['background-color: #ffe5e5; font-weight: bold;' if flag else ''] * len(row)

    styled = (df_show.style
              .apply(highlight_row, axis=1)
              .background_gradient(subset=[f'% EXT M-2 ({m2})', f'% EXT M-1 ({m1})'],
                                   cmap='YlOrRd', vmin=0, vmax=100)
              .format({f'% EXT M-2 ({m2})': '{:.0f}%',
                       f'% EXT M-1 ({m1})': '{:.0f}%',
                       'Vol. M-2+M-1 (pcs)': lambda x: f"{int(x):,}".replace(",", ".") if pd.notna(x) else '—'},
                      na_rep='—')
              .set_caption(f"Sinal de risco prospectivo — destaque: M-1 > {thr_m1}% OU M-2 > {thr_m2}%")
              .set_table_styles([{'selector': 'caption',
                                   'props': [('caption-side', 'top'),
                                             ('font-size', '14px'),
                                             ('font-weight', 'bold'),
                                             ('padding', '8px')]}]))
    return styled


def build_janela_operacional(supplier_mes, ref_date, ref_month):
    """Tabela para mes_alvo ∈ {M, M+1}: pct corrente + dias até início."""
    M = ref_month
    M_mais_1 = ref_month + 1

    df = supplier_mes[supplier_mes['mes_alvo'].isin([M, M_mais_1])].copy()
    df['mes_alvo_start'] = df['mes_alvo'].apply(lambda p: pd.Period(p, freq='M').start_time)
    df['dias_ate_mes_alvo'] = (df['mes_alvo_start'] - ref_date).dt.days
    df['mes_label'] = np.where(df['mes_alvo'] == M, f'M ({M})', f'M+1 ({M_mais_1})')
    df = df[['supplier_name', 'mes_label', 'mes_alvo', 'vol_original',
             'current_pct_ext_any', 'peak_pct_ext_any',
             'dias_ate_mes_alvo', 'antecipacao_d', 'last_snapshot']]
    return df.sort_values(['mes_alvo', 'current_pct_ext_any'], ascending=[True, False]), M, M_mais_1


def build_alerta_janela(df, m, m1, thr_pct=20, thr_lead=45):
    """Lista de fornecedores com flag de alerta por ciclo (M e M+1)."""
    df = df.copy()
    df['alerta'] = ((df['current_pct_ext_any'] > thr_pct)
                    & (df['dias_ate_mes_alvo'] < thr_lead))

    pivot = df.pivot_table(
        index='supplier_name', columns='mes_label',
        values=['current_pct_ext_any', 'alerta', 'vol_original'],
        aggfunc='first'
    )
    pivot.columns = [f'{v}|{c}' for v, c in pivot.columns]

    out = pd.DataFrame({'supplier_name': pivot.index})
    col_m  = f'M ({m})'
    col_m1 = f'M+1 ({m1})'

    for col, label in [(col_m, 'M'), (col_m1, 'M+1')]:
        pct_k  = f'current_pct_ext_any|{col}'
        flag_k = f'alerta|{col}'
        vol_k  = f'vol_original|{col}'
        out[f'pct_{label}']     = pivot.get(pct_k,  pd.Series(dtype=float)).values
        out[f'alerta_{label}']  = pivot.get(flag_k, pd.Series(dtype=float)).fillna(False).astype(bool).values
        out[f'vol_{label}']     = pivot.get(vol_k,  pd.Series(dtype=float)).values

    # Ordena: primeiro quem tem alerta em qualquer ciclo, depois por vol M
    out['_any_alert'] = out['alerta_M'] | out['alerta_M+1']
    vol_sort = out[f'vol_M'].fillna(out[f'vol_M+1']).fillna(0)
    out = out.assign(_vol_sort=vol_sort).sort_values(
        ['_any_alert', '_vol_sort'], ascending=[False, False]
    ).drop(columns=['_any_alert', '_vol_sort'])
    out = out.reset_index(drop=True)
    return out


def style_alerta_janela(df, m, m1, thr_pct, thr_lead):
    """Pandas Styler: lista limpa com ícone de alerta por ciclo."""
    # Colunas de exibição
    display = df.copy()
    display[f'% EXT M ({m})']   = display['pct_M'].apply(
        lambda x: f'{x:.0f}%' if pd.notna(x) else '—')
    display[f'% EXT M+1 ({m1})'] = display['pct_M+1'].apply(
        lambda x: f'{x:.0f}%' if pd.notna(x) else '—')
    display['Vol M (pcs)'] = display['vol_M'].apply(
        lambda x: f"{int(x):,}".replace(',','.') if pd.notna(x) else '—')
    display['Vol M+1 (pcs)'] = display['vol_M+1'].apply(
        lambda x: f"{int(x):,}".replace(',','.') if pd.notna(x) else '—')
    display[f'Alerta M ({m})']   = display['alerta_M'].map({True:  '🔴 ALERTA', False: '✅ OK'})
    display[f'Alerta M+1 ({m1})'] = display['alerta_M+1'].map({True: '🔴 ALERTA', False: '✅ OK'})

    cols_show = ['supplier_name',
                 f'% EXT M ({m})',   f'Alerta M ({m})',
                 f'% EXT M+1 ({m1})', f'Alerta M+1 ({m1})',
                 'Vol M (pcs)', 'Vol M+1 (pcs)']
    display = display[cols_show].rename(columns={'supplier_name': 'Fornecedor'})

    def row_color(row):
        base = [''] * len(row)
        alert_m  = row.get(f'Alerta M ({m})',   '') == '🔴 ALERTA'
        alert_m1 = row.get(f'Alerta M+1 ({m1})', '') == '🔴 ALERTA'
        if alert_m or alert_m1:
            base = ['background-color: #fff0f0;'] * len(row)
        if alert_m:
            idx = list(row.index).index(f'Alerta M ({m})')
            base[idx] = 'background-color: #ffe0e0; font-weight: bold;'
        if alert_m1:
            idx = list(row.index).index(f'Alerta M+1 ({m1})')
            base[idx] = 'background-color: #ffe0e0; font-weight: bold;'
        return base

    styled = (display.style
              .apply(row_color, axis=1)
              .set_caption(
                  f'Janela operacional — Alerta: pct_ext_any > {thr_pct}% '
                  f'com menos de {thr_lead} dias até o mes_alvo')
              .set_table_styles([
                  {'selector': 'caption',
                   'props': [('caption-side','top'),('font-size','14px'),
                             ('font-weight','bold'),('padding','8px')]},
                  {'selector': 'th',
                   'props': [('text-align','center')]},
                  {'selector': 'td',
                   'props': [('text-align','center')]},
                  {'selector': 'td:first-child, th:first-child',
                   'props': [('text-align','left')]},
              ])
              .hide(axis='index'))
    return styled


# 3. Seção 1 — Cascateamento histórico

**Definição:** `Cascade(supplier) = P(peak_{M+1} > 30% | peak_M > 30%)`.

Considera todos os pares consecutivos de `mes_alvo` no histórico do fornecedor. Mesmo critério usado no diagnóstico (README, Seção 2).

**Destaque negativo:** fornecedores com cascade > 50%.

In [164]:
cascade_hist = build_cascade_table(supplier_mes, threshold=PEAK_THRESHOLD_CASCADE)
print(f"Fornecedores com pares consecutivos avaliáveis: "
      f"{cascade_hist['n_pares_M_problematicos'].gt(0).sum()}")
cascade_hist.head(15)


Fornecedores com pares consecutivos avaliáveis: 29


,supplier_name,vol_total,n_mes_alvo,pct_ext_medio,n_pares_M_problematicos,cascade_pct
19,INTERTEXTIL,186146,6,93.3,5,100.0
12,FABIO,155221,6,24.7,2,100.0
23,LUTESTIL,147127,5,72.3,3,100.0
8,CLESTE BRASIL CONFECÇÕES LTDA,53575,4,62.6,2,100.0
3,AZZURRA,33825,6,38.4,2,100.0
30,NATURAL COMPANY CONFECCOES LTDA,19166,2,100.0,1,100.0
20,KABRIOLLI,15445,5,54.8,1,100.0
21,LATINA,10050,2,100.0,1,100.0
37,WARUSKY,9654,2,100.0,1,100.0
7,CLARA BELLA,8414,5,38.9,1,100.0


In [165]:
style_cascade_hist(cascade_hist, threshold=CASCADE_THRESHOLD_HIST)


,Fornecedor,Vol. total (pcs),N° mes_alvo,% EXT médio (peak ponderado),N° pares M problemáticos,Cascade %
19,INTERTEXTIL,186.146,6,93%,5,100%
12,FABIO,155.221,6,25%,2,100%
23,LUTESTIL,147.127,5,72%,3,100%
8,CLESTE BRASIL CONFECÇÕES LTDA,53.575,4,63%,2,100%
3,AZZURRA,33.825,6,38%,2,100%
30,NATURAL COMPANY CONFECCOES LTDA,19.166,2,100%,1,100%
20,KABRIOLLI,15.445,5,55%,1,100%
21,LATINA,10.050,2,100%,1,100%
37,WARUSKY,9.654,2,100%,1,100%
7,CLARA BELLA,8.414,5,39%,1,100%


In [166]:
# Visão gráfica complementar (ordenado por cascade)
df_plot = cascade_hist.dropna(subset=['cascade_pct']).copy()
df_plot['flag_alto'] = np.where(df_plot['cascade_pct'] > CASCADE_THRESHOLD_HIST,
                                f'> {CASCADE_THRESHOLD_HIST}%',
                                f'≤ {CASCADE_THRESHOLD_HIST}%')
fig = px.bar(df_plot.sort_values('cascade_pct'),
             x='cascade_pct', y='supplier_name', color='flag_alto',
             color_discrete_map={f'> {CASCADE_THRESHOLD_HIST}%': '#d62728',
                                 f'≤ {CASCADE_THRESHOLD_HIST}%': '#bdbdbd'},
             orientation='h',
             title=f'Cascateamento histórico por fornecedor (threshold de risco: {CASCADE_THRESHOLD_HIST}%)',
             labels={'cascade_pct': 'Cascade %', 'supplier_name': 'Fornecedor',
                     'flag_alto': 'Risco'},
             text='cascade_pct',
             height=max(350, 18 * len(df_plot)))
fig.update_traces(texttemplate='%{text:.0f}%', textposition='outside')
fig.update_layout(template='plotly_white', xaxis_range=[0, 110])
fig.show()

# 4. Seção 2 — Sinal de risco prospectivo (M-1 e M-2)

**Hipótese (do diagnóstico):** alta `pct_ext_any` em ciclos recentes é um forte indicador de cascateamento nos próximos ciclos. Um fornecedor que terminou M-1 com pct alto tende a entregar M com pct alto também.

**Critério de destaque negativo:**
- `pct_ext_any` em **M-1 > 60%** **ou**
- `pct_ext_any` em **M-2 > 70%**

> Para o snapshot retrospectivo usamos o **peak** de `pct_ext_any` ao longo da vida do ciclo (mesma definição usada no diagnóstico — evita reversões pontuais que mascarariam o problema).

In [167]:
pct_recente, M_1, M_2 = build_pct_recente_table(supplier_mes, REF_MONTH)
print(f"M-1 = {M_1}  |  M-2 = {M_2}")
pct_recente.head()


M-1 = 2026-03  |  M-2 = 2026-02


,M-1 (2026-03),M-2 (2026-02),vol_total_M-2_M-1
supplier_name,,,
BAE BRASIL,81.4,87.8,227581.0
BY COTTON,42.6,84.4,88240.0
ART LIVRE,89.4,96.7,85861.0
MALHAS D'STEFANO,21.1,18.7,82185.0
RIZLLEP,100.0,0.0,27067.0


In [168]:
style_pct_recente(pct_recente, M_1, M_2,
                  thr_m1=PCT_M_1_THRESHOLD, thr_m2=PCT_M_2_THRESHOLD)


,% EXT M-2 (2026-02),% EXT M-1 (2026-03),Vol. M-2+M-1 (pcs)
supplier_name,,,
BAE BRASIL,88%,81%,227.581
BY COTTON,84%,43%,88.240
ART LIVRE,97%,89%,85.861
MALHAS D'STEFANO,19%,21%,82.185
RIZLLEP,0%,100%,27.067
DDAL,100%,56%,25.290
RDM,14%,70%,18.270
MAURA,100%,92%,15.593
LUTESTIL,100%,—,12.398


In [169]:
# Lista de fornecedores em alerta
col_m1 = f'M-1 ({M_1})'
col_m2 = f'M-2 ({M_2})'
alert_recente = pct_recente.copy()
alert_recente['flag'] = (
    (alert_recente.get(col_m1, np.nan) > PCT_M_1_THRESHOLD)
    | (alert_recente.get(col_m2, np.nan) > PCT_M_2_THRESHOLD)
)
alert_list = alert_recente[alert_recente['flag']].drop(columns='flag')
print(f"Fornecedores em alerta (M-1 > {PCT_M_1_THRESHOLD}% ou M-2 > {PCT_M_2_THRESHOLD}%): {len(alert_list)}")
alert_list

Fornecedores em alerta (M-1 > 60% ou M-2 > 70%): 18


,M-1 (2026-03),M-2 (2026-02),vol_total_M-2_M-1
supplier_name,,,
BAE BRASIL,81.4,87.8,227581.0
BY COTTON,42.6,84.4,88240.0
ART LIVRE,89.4,96.7,85861.0
RIZLLEP,100.0,0.0,27067.0
DDAL,56.2,100.0,25290.0
RDM,69.5,14.0,18270.0
MAURA,91.5,100.0,15593.0
LUTESTIL,NaN,100.0,12398.0
"Ges Confecção, Comercio e Serviços de Serigrafia LTDA",42.5,100.0,12079.0


# 5. Seção 3 — Janela operacional dos ciclos ativos (M e M+1)

**Lógica:** para os ciclos cujo `mes_alvo ∈ {M, M+1}`, queremos identificar fornecedores que já estão alterando muito (`pct_ext_any > 20%`) com o ciclo dentro do lead time produtivo (< 45 dias até o início do `mes_alvo`).

Esses são os casos sem janela suficiente para o time atuar — toda alteração futura cai no modo reativo.

**Antecipação:** dias entre **data de referência** e o início de `mes_alvo`. Aqui o número é **prospectivo** (quanto resta), diferente da Seção 1 do diagnóstico (onde era retrospectivo).

In [170]:
janela, M, M_mais_1 = build_janela_operacional(supplier_mes, REF_DATE, REF_MONTH)
print(f"M = {M}  |  M+1 = {M_mais_1}")
print(f"Pares (supplier × mes_alvo) ativos: {len(janela)}")
janela.head(20)


M = 2026-04  |  M+1 = 2026-05
Pares (supplier × mes_alvo) ativos: 37


,supplier_name,mes_label,mes_alvo,vol_original,current_pct_ext_any,peak_pct_ext_any,dias_ate_mes_alvo,antecipacao_d,last_snapshot
156,MASH,M (2026-04),2026-04,14199,49.5,99.6,-29,23.0,2026-04-27
37,BAE BRASIL,M (2026-04),2026-04,152188,31.8,97.6,-29,65.0,2026-04-27
166,MAURA,M (2026-04),2026-04,3474,27.4,100.0,-29,58.0,2026-04-27
11,ART LIVRE,M (2026-04),2026-04,48701,27.0,71.5,-29,72.0,2026-04-27
1,ABBA,M (2026-04),2026-04,11747,24.2,45.4,-29,65.0,2026-04-27
203,RDM,M (2026-04),2026-04,10158,21.4,42.9,-29,16.0,2026-04-27
86,DDAL,M (2026-04),2026-04,17050,17.5,34.9,-29,65.0,2026-04-27
109,"Ges Confecção, Comercio e Serviços de Serigra...",M (2026-04),2026-04,12096,16.8,82.9,-29,30.0,2026-04-27
48,BY COTTON,M (2026-04),2026-04,10487,14.3,14.3,-29,65.0,2026-04-27
193,PIXIE,M (2026-04),2026-04,2138,5.5,94.4,-29,37.0,2026-04-27


In [171]:
alert_janela = build_alerta_janela(
    janela, M, M_mais_1, thr_pct=PCT_ATIVO_THRESHOLD, thr_lead=LEAD_TIME_DIAS
)
style_alerta_janela(alert_janela, M, M_mais_1, PCT_ATIVO_THRESHOLD, LEAD_TIME_DIAS)


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_47931/691321623.py:164: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[f'alerta_{label}']  = pivot.get(flag_k, pd.Series(dtype=float)).fillna(False).astype(bool).values
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_47931/691321623.py:164: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[f'alerta_{label}']  = pivot.get(flag_k, pd.Series(dtype=float)).fillna(False).astype(bool).values


Fornecedor,% EXT M (2026-04),Alerta M (2026-04),% EXT M+1 (2026-05),Alerta M+1 (2026-05),Vol M (pcs),Vol M+1 (pcs)
BAE BRASIL,32%,🔴 ALERTA,19%,✅ OK,152.188,100.200
ART LIVRE,27%,🔴 ALERTA,9%,✅ OK,48.701,32.113
DDAL,18%,✅ OK,25%,🔴 ALERTA,17.050,14.834
MASH,50%,🔴 ALERTA,0%,✅ OK,14.199,8.555
"Ges Confecção, Comercio e Serviços de Serigrafia LTDA",17%,✅ OK,50%,🔴 ALERTA,12.096,2.005
ABBA,24%,🔴 ALERTA,20%,✅ OK,11.747,8.951
BY COTTON,14%,✅ OK,100%,🔴 ALERTA,10.487,778
RDM,21%,🔴 ALERTA,27%,🔴 ALERTA,10.158,10.217
MAURA,27%,🔴 ALERTA,26%,🔴 ALERTA,3.474,7.110
GOAT,0%,✅ OK,100%,🔴 ALERTA,3.059,1.541


In [172]:
# Resumo de alertas ativos
n_m  = alert_janela['alerta_M'].sum()
n_m1 = alert_janela['alerta_M+1'].sum()
print(f"Alertas ativos em M  ({M}) : {n_m}  fornecedor(es)")
print(f"Alertas ativos em M+1 ({M_mais_1}): {n_m1} fornecedor(es)")
print()
print("Fornecedores em ALERTA em M:")
print(alert_janela.loc[alert_janela['alerta_M'], 'supplier_name'].tolist())
print()
print("Fornecedores em ALERTA em M+1:")
print(alert_janela.loc[alert_janela['alerta_M+1'], 'supplier_name'].tolist())


Alertas ativos em M  (2026-04) : 6  fornecedor(es)
Alertas ativos em M+1 (2026-05): 7 fornecedor(es)

Fornecedores em ALERTA em M:
['BAE BRASIL', 'ART LIVRE', 'MASH', 'ABBA', 'RDM', 'MAURA']

Fornecedores em ALERTA em M+1:
['DDAL', 'Ges Confecção, Comercio e Serviços  de Serigrafia LTDA', 'BY COTTON', 'RDM', 'MAURA', 'GOAT', 'KABRIOLLI']


In [173]:
# Exportar lista de alertas
alert_janela.to_csv(
    Path(output_path) / 'alerta_risco_cadeia' / f'janela_operacional_M_M+1_{today}.csv',
    index=False
)
print("Exportado.")


Exportado.


# 6. Seção 4 — Aumento de alteração no ciclo corrente (M)

**Crivo:** houve **qualquer** evento de aumento de `pct_ext_any` week-over-week dentro do ciclo M — independentemente de quando foi.

Um fornecedor que aumentou a alteração em qualquer semana do ciclo (mesmo que depois tenha estabilizado) está sinalizando deterioração ativa. O gatilho não é o estado atual, é a **ocorrência** do evento.

**Coluna Δ% máx:** maior aumento semanal registrado no ciclo, para contextualizar a magnitude.  
**Volume de referência:** alocação em ciclos Base + Extra para `mes_alvo = M`, ordenada de maior para menor.

In [174]:
DELTA_THRESHOLD_M = 20  # pp — mínimo de aumento semanal para disparar alerta na Seção 4


def build_delta_semanal_m(df_evol, ref_month, delta_thr=DELTA_THRESHOLD_M):
    """
    Para mes_alvo == M (Base + Extra):
    - aumentando = True se houve QUALQUER semana com delta pct_ext_any > delta_thr no ciclo
    - delta_max  = maior aumento semanal registrado (para contexto de magnitude)
    """
    m = ref_month
    df_m = df_evol[df_evol['mes_alvo'] == m].copy()
    df_be = df_m[df_m['cycle_type'].isin(['Base', 'Extra'])]
    if not df_be.empty:
        df_m = df_be

    g = (df_m.groupby(['supplier_name', 'snapshot_week'])
         .agg(vol_original=('vol_original', 'sum'),
              vol_ext_any=('vol_ext_any', 'sum'))
         .reset_index())
    g['pct_ext_any'] = np.where(g['vol_original'] > 0,
                                g['vol_ext_any'] / g['vol_original'] * 100, 0)

    rows = []
    for sup, grp in g.groupby('supplier_name'):
        grp = grp.sort_values('snapshot_week')
        vol_total = int(grp['vol_original'].iloc[0])
        pct_atual = grp.iloc[-1]['pct_ext_any']

        # Todos os deltas week-over-week dentro do ciclo M
        deltas = grp['pct_ext_any'].diff().dropna()
        significativos = deltas[deltas > delta_thr]

        houve_aumento = len(significativos) > 0
        delta_max = round(significativos.max(), 1) if houve_aumento else np.nan

        rows.append({
            'supplier_name': sup,
            'vol_m':         vol_total,
            'pct_atual':     round(pct_atual, 1),
            'delta_max':     delta_max,
            'n_snapshots':   len(grp),
            'aumentando':    houve_aumento,
        })

    return pd.DataFrame(rows).sort_values('vol_m', ascending=False).reset_index(drop=True)


def style_delta_semanal(df, m, delta_thr=DELTA_THRESHOLD_M):
    """Styler: SIM/NÃO com destaque nos que tiveram aumento > delta_thr pp em alguma semana."""
    display = df.copy()
    display['Crivo'] = display['aumentando'].map({True: '🔴 SIM', False: '✅ NÃO'})
    display['% EXT atual'] = display['pct_atual'].apply(
        lambda x: f'{x:.0f}%' if pd.notna(x) else '—')
    display['Δ% máx (semana)'] = display['delta_max'].apply(
        lambda x: f'+{x:.0f}%' if pd.notna(x) else '—')
    display['Vol. M (pcs)'] = display['vol_m'].apply(
        lambda x: f"{int(x):,}".replace(',', '.'))

    cols = ['supplier_name', 'Vol. M (pcs)', '% EXT atual', 'Δ% máx (semana)', 'Crivo']
    display = display[cols].rename(columns={'supplier_name': 'Fornecedor'})

    def row_color(row):
        if row['Crivo'] == '🔴 SIM':
            return (['background-color: #fff0f0;'] * (len(row) - 1)
                    + ['background-color: #ffe0e0; font-weight: bold;'])
        return [''] * len(row)

    return (display.style
            .apply(row_color, axis=1)
            .set_caption(
                f'Aumento de alteração no ciclo corrente M ({m}) — '
                f'evento week-over-week > {delta_thr} pp — Base + Extra')
            .set_table_styles([
                {'selector': 'caption',
                 'props': [('caption-side', 'top'), ('font-size', '14px'),
                           ('font-weight', 'bold'), ('padding', '8px')]},
                {'selector': 'th, td', 'props': [('text-align', 'center')]},
                {'selector': 'td:first-child, th:first-child',
                 'props': [('text-align', 'left')]},
            ])
            .hide(axis='index'))


delta_m = build_delta_semanal_m(df_evolucao, REF_MONTH)
print(f"Fornecedores com aumento > {DELTA_THRESHOLD_M} pp em alguma semana de M ({REF_MONTH}): "
      f"{delta_m['aumentando'].sum()} de {len(delta_m)}")
style_delta_semanal(delta_m, REF_MONTH)

Fornecedores com aumento > 20 pp em alguma semana de M (2026-04): 15 de 20


Fornecedor,Vol. M (pcs),% EXT atual,Δ% máx (semana),Crivo
BAE BRASIL,152.188,38%,+75%,🔴 SIM
ART LIVRE,48.701,26%,+56%,🔴 SIM
MALHAS D'STEFANO,26.341,0%,+25%,🔴 SIM
Lunelli Nordeste,20.000,10%,—,✅ NÃO
DDAL,17.050,20%,+26%,🔴 SIM
RIZLLEP,17.028,2%,+100%,🔴 SIM
MASH,14.199,50%,+100%,🔴 SIM
"Ges Confecção, Comercio e Serviços de Serigrafia LTDA",12.096,12%,+45%,🔴 SIM
ABBA,11.747,26%,+31%,🔴 SIM
BY COTTON,10.487,8%,—,✅ NÃO


# 7. Consolidação — Fornecedores em múltiplos crivos

Cruzamento dos quatro alertas para identificar fornecedores que aparecem em mais de um crivo (risco composto).

In [175]:
set_cascade = set(cascade_hist.loc[cascade_hist['cascade_pct'] > CASCADE_THRESHOLD_HIST, 'supplier_name'])
set_recente = set(alert_list.index)
set_op      = set(alert_janela.loc[
    alert_janela['alerta_M'] | alert_janela['alerta_M+1'], 'supplier_name'])
set_delta   = set(delta_m.loc[delta_m['aumentando'], 'supplier_name'])

todos = sorted(set_cascade | set_recente | set_op | set_delta)
consol = pd.DataFrame({'supplier_name': todos})
consol['Cascade hist > 50%']        = consol['supplier_name'].isin(set_cascade)
consol['M-1 ou M-2 acima limiar']   = consol['supplier_name'].isin(set_recente)
consol['Janela operacional (M/M+1)'] = consol['supplier_name'].isin(set_op)
consol['Aumento em M (semana)']     = consol['supplier_name'].isin(set_delta)

crivo_cols = ['Cascade hist > 50%', 'M-1 ou M-2 acima limiar',
              'Janela operacional (M/M+1)', 'Aumento em M (semana)']
consol['n_crivos'] = consol[crivo_cols].sum(axis=1)
consol = consol.sort_values('n_crivos', ascending=False)

def fmt_check(v):
    return '🔴' if v else ''

(consol.style
    .format({c: fmt_check for c in crivo_cols})
    .background_gradient(subset=['n_crivos'], cmap='Reds', vmin=0, vmax=4)
    .set_caption('Consolidação — fornecedores por número de crivos em alerta')
    .hide(axis='index'))

supplier_name,Cascade hist > 50%,M-1 ou M-2 acima limiar,Janela operacional (M/M+1),Aumento em M (semana),n_crivos
BAE BRASIL,🔴,🔴,🔴,🔴,4
MAURA,🔴,🔴,🔴,🔴,4
"Ges Confecção, Comercio e Serviços de Serigrafia LTDA",🔴,🔴,🔴,🔴,4
ART LIVRE,🔴,🔴,🔴,🔴,4
INDÚSTRIA TEXTIL BETILHA LTDA,🔴,🔴,,🔴,3
RIZLLEP,🔴,🔴,,🔴,3
RDM,,🔴,🔴,🔴,3
BY COTTON,🔴,🔴,🔴,,3
PIXIE,🔴,🔴,,🔴,3
DALOP,🔴,🔴,,🔴,3


# 8. Export

In [176]:
out_dir = Path(output_path) / 'alerta_risco_cadeia'
out_dir.mkdir(parents=True, exist_ok=True)

cascade_hist.to_csv(out_dir / f'cascade_historico_{today}.csv', index=False)
pct_recente.to_csv(out_dir / f'pct_ext_any_M-1_M-2_{today}.csv')
janela.to_csv(out_dir / f'janela_operacional_M_M+1_{today}.csv', index=False)
delta_m.to_csv(out_dir / f'delta_semanal_M_{today}.csv', index=False)
consol.to_csv(out_dir / f'consolidacao_alertas_{today}.csv', index=False)

print(f"Arquivos exportados em: {out_dir}")
for f in sorted(out_dir.glob(f'*_{today}.csv')):
    print(f"  {f.name}")

Arquivos exportados em: ../../outputs/alerta_risco_cadeia
  cascade_historico_20260430.csv
  consolidacao_alertas_20260430.csv
  delta_semanal_M_20260430.csv
  janela_operacional_M_M+1_20260430.csv
  pct_ext_any_M-1_M-2_20260430.csv
